In [5]:
# Run these commands in Anaconda Prompt or a terminal:
# 1. Navigate to the repository folder:
#    cd data_processing
# 2. Create the environment:
#    conda env create -f environment.yml
# 3. Activate it:
#    conda activate cs424env
# 4. Open the notebook:
#    jupyter notebook visulas.ipynb
# 5. Use the kernel for cs424env and run all cells from top to bottom.

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import altair as alt


In [6]:
# Load our data
data = pd.read_csv("../Updated_Dataset.csv")
locations = pd.read_csv("locations.csv")
fixes = pd.read_csv("fixes.csv")

In [7]:
# Blank out the values we know are wrong
for visit, column in zip(fixes["visit_id"], fixes["column"]):
    data.loc[data["visit_id"] == visit, column] = None

# Recalculate these instead of trusting the spreadsheet formulas
data["available_chairs"] = data["total_chairs"] - data["occupied_chairs"]
data["available_tables"] = data["total_tables"] - data["occupied_tables"]
data["chair_occupancy_pct"] = data["occupied_chairs"] / data["total_chairs"] * 100
data["table_occupancy_pct"] = data["occupied_tables"] / data["total_tables"] * 100
data = data.drop(columns=["chair_occupancy_rate", "table_occupancy_rate"])

# Print any visits that are wrong
problems = data[data["occupied_tables"] > data["occupied_chairs"]]
print("Visits with more occupied tables than chairs:", list(problems["visit_id"]))

# Add new columns
data = data.merge(locations, on="location")
data["session_id"] = data["date"] + " " + data["time_of_day"]
data["table_chair_gap"] = data["table_occupancy_pct"] - data["chair_occupancy_pct"]
data["people_per_table"] = data["occupied_chairs"] / data["occupied_tables"]
data["has_open_table"] = (data["available_tables"] > 0).astype(float)
data.loc[data["available_tables"].isna(), "has_open_table"] = None
data.round(2).to_csv("../processed_data/observations_clean.csv", index=False)

Visits with more occupied tables than chairs: []


In [8]:
# Q1: average occupancy for each lounge at each time of day
q1 = data.groupby(["location_label", "time_of_day"])[["chair_occupancy_pct", "table_occupancy_pct"]].mean()
q1.round(2).to_csv("../processed_data/q1_by_time_of_day.csv")

# Q2: the same lounge, day and time of day, on a rainy day vs a sunny day
q2 = data.pivot_table(index=["location_label", "day", "time_of_day"], columns="weather",
    values="chair_occupancy_pct")[["Rain", "Sunny"]].dropna()
q2.round(2).to_csv("../processed_data/q2_rain_vs_sun.csv")

# Q4 and our new question: open seats, open tables and people per table for each lounge
q4 = data.groupby("location_label")[["chair_occupancy_pct", "table_occupancy_pct", "available_chairs",
    "has_open_table", "people_per_table"]].mean()
q4.round(2).sort_values("chair_occupancy_pct").to_csv("../processed_data/q4_by_location.csv")

# How many visits we have for each day and time of day
days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]
times = ["Morning", "Midday", "Afternoon"]
coverage = pd.crosstab(data["day"], data["time_of_day"]).reindex(index=days, columns=times, fill_value=0)
coverage.to_csv("../processed_data/coverage.csv")
periods = ["Morning", "Midday", "Afternoon"]

In [9]:
coverage_data = coverage.rename_axis(
    index="day", columns="time_of_day"
).stack().reset_index(name="visits")

base = alt.Chart(coverage_data).encode(
    x=alt.X("time_of_day:O", sort=times, title="Time of day"),
    y=alt.Y("day:O", sort=days, title="Weekday")
)

cells = base.mark_rect().encode(
    color=alt.Color("visits:Q", scale=alt.Scale(scheme="blues"), title="Visits")
)

labels = base.mark_text().encode(
    text="visits:Q",
    color=alt.condition(
        alt.datum.visits > coverage_data["visits"].max() / 2,
        alt.value("white"),
        alt.value("black")
    )
)

(cells + labels).properties(
    width=400,
    height=250,
    title="Number of visits"
)

alt.LayerChart(...)

In [10]:
subset = data[
    (data["location_label"] == "West Terrace")
    & (data["time_of_day"] == "Morning")
    & (data["weather"].isin(["Sunny", "Rain"]))
]

alt.Chart(subset).mark_bar().encode(
    x=alt.X("weather:N", title="Weather", sort=["Sunny", "Rain"]),
    y=alt.Y(
        "mean(chair_occupancy_pct):Q",
        title="Average chair occupancy (%)",
        scale=alt.Scale(domain=[0, 100])
    ),
    tooltip=[
        alt.Tooltip("mean(chair_occupancy_pct):Q", format=".1f"),
        alt.Tooltip("valid(chair_occupancy_pct):Q", title="Valid visits")
    ]
).properties(
    width=350,
    height=300,
    title="West Terrace: morning occupancy by weather"
)

alt.Chart(...)

In [11]:
subset = data[data["location_label"] == "Montgomery Lounge"]

alt.Chart(subset).mark_point(filled=True, size=70).encode(
    x=alt.X(
        "chair_occupancy_pct:Q",
        title="Chair occupancy (%)",
        scale=alt.Scale(domain=[0, 100])
    ),
    y=alt.Y(
        "table_occupancy_pct:Q",
        title="Table occupancy (%)",
        scale=alt.Scale(domain=[0, 100])
    ),
    color=alt.Color("time_of_day:N", title="Time of day", sort=periods),
    shape=alt.Shape("time_of_day:N", title="Time of day", sort=periods),
    tooltip=["visit_id", "chair_occupancy_pct", "table_occupancy_pct"]
).properties(
    width=400,
    height=400,
    title="Montgomery Lounge: tables versus chairs"
)

alt.Chart(...)

In [12]:
alt.Chart(data).mark_line(point=True).encode(
    x=alt.X(
        "time_of_day:O",
        title="Time of day",
        sort=periods,
        axis=alt.Axis(labelAngle=0)
    ),
    y=alt.Y(
        "mean(chair_occupancy_pct):Q",
        title="Average chair occupancy (%)",
        scale=alt.Scale(domain=[0, 100])
    ),
    color=alt.Color("location_label:N", title="Study area")
).properties(
    width=600,
    height=350,
    title="Chair occupancy throughout the day"
)

alt.Chart(...)

In [13]:
alt.Chart(data).mark_line(point=True).encode(
    x=alt.X(
        "time_of_day:O",
        title="Time of day",
        sort=periods,
        axis=alt.Axis(labelAngle=0)
    ),
    y=alt.Y(
        "mean(available_tables):Q",
        title="Average empty tables",
        scale=alt.Scale(zero=True)
    ),
    color=alt.Color("location_label:N", title="Study area")
).properties(
    width=600,
    height=350,
    title="Empty tables throughout the day"
)

alt.Chart(...)

In [14]:
alt.Chart(data).mark_bar().encode(
    x=alt.X(
        "location_label:N",
        title="Study area",
        axis=alt.Axis(labelAngle=-35)
    ),
    xOffset=alt.XOffset("time_of_day:N", sort=periods),
    y=alt.Y(
        "mean(has_open_table):Q",
        title="Visits with an empty table",
        scale=alt.Scale(domain=[0, 1]),
        axis=alt.Axis(format=".0%")
    ),
    color=alt.Color("time_of_day:N", title="Time of day", sort=periods),
    tooltip=[
        "location_label",
        "time_of_day",
        alt.Tooltip("mean(has_open_table):Q", title="Frequency", format=".0%"),
        alt.Tooltip("valid(has_open_table):Q", title="Valid visits")
    ]
).properties(
    width=700,
    height=350,
    title="How often was an empty table available?"
)

alt.Chart(...)

In [15]:
occupancy_comparison = data.melt(
    id_vars=["location_label"],
    value_vars=["chair_occupancy_pct", "table_occupancy_pct"],
    var_name="occupancy_type",
    value_name="occupancy"
)

occupancy_comparison["occupancy_type"] = (
    occupancy_comparison["occupancy_type"].replace({
        "chair_occupancy_pct": "Chairs",
        "table_occupancy_pct": "Tables"
    })
)

bar_chart = alt.Chart(occupancy_comparison).mark_bar().encode(
    x=alt.X(
        "location_label:N",
        title="Study area",
        axis=alt.Axis(labelAngle=-35)
    ),
    xOffset="occupancy_type:N",
    y=alt.Y(
        "mean(occupancy):Q",
        title="Average occupancy (%)",
        scale=alt.Scale(domain=[0, 100])
    ),
    color=alt.Color("occupancy_type:N", title="Occupancy type"),
    tooltip=[
        "location_label:N",
        "occupancy_type:N",
        alt.Tooltip("mean(occupancy):Q", title="Average (%)", format=".1f"),
        alt.Tooltip("valid(occupancy):Q", title="Valid visits")
    ]
).properties(
    width=700,
    height=350,
    title="Average table and chair occupancy by location"
)

bar_chart

alt.Chart(...)

In [16]:
heatmap_data = (
    data.groupby(["location_label", "time_of_day"])["chair_occupancy_pct"]
    .agg(average_occupancy="mean", observations="count")
    .reset_index()
)

base = alt.Chart(heatmap_data).encode(
    x=alt.X(
        "time_of_day:O",
        sort=periods,
        title="Time of day",
        axis=alt.Axis(labelAngle=0)
    ),
    y=alt.Y("location_label:N", title="Study area")
)

cells = base.mark_rect().encode(
    color=alt.Color(
        "average_occupancy:Q",
        title="Occupancy (%)",
        scale=alt.Scale(domain=[0, 100], scheme="blues")
    ),
    tooltip=[
        "location_label:N",
        "time_of_day:O",
        alt.Tooltip("average_occupancy:Q", title="Average (%)", format=".1f"),
        alt.Tooltip("observations:Q", title="Valid visits")
    ]
)

labels = base.mark_text().encode(
    text=alt.Text("average_occupancy:Q", format=".1f"),
    color=alt.condition(
        alt.datum.average_occupancy > 60,
        alt.value("white"),
        alt.value("black")
    )
)

heatmap_chart = (cells + labels).properties(
    width=450,
    height=350,
    title="Average chair occupancy by location and time"
)

heatmap_chart

alt.LayerChart(...)

In [17]:
scatter_chart = alt.Chart(data).mark_point(
    filled=True,
    opacity=0.65,
    size=55
).encode(
    x=alt.X(
        "chair_occupancy_pct:Q",
        title="Chair occupancy (%)",
        scale=alt.Scale(domain=[0, 100])
    ),
    y=alt.Y(
        "table_occupancy_pct:Q",
        title="Table occupancy (%)",
        scale=alt.Scale(domain=[0, 100])
    ),
    color=alt.Color("time_of_day:N", title="Time of day", sort=periods),
    tooltip=[
        "visit_id:N",
        "location_label:N",
        "time_of_day:N",
        alt.Tooltip("chair_occupancy_pct:Q", title="Chairs (%)", format=".1f"),
        alt.Tooltip("table_occupancy_pct:Q", title="Tables (%)", format=".1f")
    ]
).properties(
    width=200,
    height=200
).facet(
    facet=alt.Facet("location_label:N", title=None),
    columns=3
).properties(
    title="Table and chair occupancy across individual visits"
)

scatter_chart

alt.FacetChart(...)

In [19]:
# Task 5: heatmap linked to a scatterplot of individual visits, with a weather filter
data["study_area"] = data["location_label"]
outdoor = data["space_type"] == "outdoor"
data.loc[outdoor, "study_area"] = data.loc[outdoor, "study_area"] + " (outdoor)"

area_order = list(data.groupby("study_area")["chair_occupancy_pct"].mean().sort_values().index)

picked_cells = alt.selection_point(fields=["study_area", "time_of_day"], on="click", clear="dblclick")

weather_choice = alt.param(
    value="All",
    bind=alt.binding_radio(options=["All", "Sunny", "Rain", "Cloudy"], name="Weather (both views): ")
)
weather_match = (weather_choice == "All") | (alt.datum.weather == weather_choice)

visits = alt.Chart(data).transform_filter(weather_match)

heatmap_base = visits.transform_aggregate(
    average_occupancy="mean(chair_occupancy_pct)",
    observations="count()",
    groupby=["study_area", "time_of_day"]
).transform_calculate(
    percent_label=alt.expr.format(alt.datum.average_occupancy, ".0f") + "%",
    visits_label=alt.expr.if_(alt.datum.observations == 1, "1 visit", alt.datum.observations + " visits")
).encode(
    x=alt.X(
        "time_of_day:O",
        title="Time of day",
        scale=alt.Scale(domain=periods),
        axis=alt.Axis(labelAngle=0, orient="top", labelFontSize=13, titleFontSize=14)
    ),
    y=alt.Y(
        "study_area:N",
        title=None,
        scale=alt.Scale(domain=area_order),
        axis=alt.Axis(labelFontSize=13)
    ),
    opacity=alt.condition(picked_cells, alt.value(1), alt.value(0.3))
)

cells = heatmap_base.mark_rect().encode(
    color=alt.Color(
        "average_occupancy:Q",
        title="Average chair occupancy (%)",
        scale=alt.Scale(domain=[0, 100], scheme="blues"),
        legend=alt.Legend(orient="bottom", direction="horizontal")
    ),
    stroke=alt.condition(picked_cells, alt.value("black"), alt.value("white"), empty=False),
    strokeWidth=alt.condition(picked_cells, alt.value(3), alt.value(1.5), empty=False),
    tooltip=[
        alt.Tooltip("study_area:N", title="Study area"),
        alt.Tooltip("time_of_day:O", title="Time of day"),
        alt.Tooltip("average_occupancy:Q", title="Average (%)", format=".1f"),
        alt.Tooltip("observations:Q", title="Visits")
    ]
).add_params(picked_cells)

labels = heatmap_base.mark_text(baseline="bottom", dy=-1, fontSize=16, fontWeight="bold").encode(
    text="percent_label:N",
    color=alt.condition(
        alt.datum.average_occupancy > 60,
        alt.value("white"),
        alt.value("black")
    )
)

count_labels = heatmap_base.mark_text(baseline="top", dy=2, fontSize=11).encode(
    text="visits_label:N",
    color=alt.condition(
        alt.datum.average_occupancy > 60,
        alt.value("white"),
        alt.value("#444444")
    )
)

empty_base = alt.Chart(data).transform_calculate(
    matches_weather=alt.expr.if_(weather_match, 1, 0)
).transform_aggregate(
    matching_visits="sum(matches_weather)",
    groupby=["study_area", "time_of_day"]
).transform_filter(
    alt.datum.matching_visits == 0
).encode(
    x=alt.X("time_of_day:O", scale=alt.Scale(domain=periods)),
    y=alt.Y("study_area:N", scale=alt.Scale(domain=area_order))
)

empty_cells = empty_base.mark_rect(color="#f2f2f2", stroke="white", strokeWidth=1.5)
empty_labels = empty_base.mark_text(fontSize=11, color="gray", text="no visits")

heatmap_chart = (empty_cells + empty_labels + cells + labels + count_labels).properties(
    width=310,
    height=330,
    title=alt.Title(
        "Average chair occupancy",
    )
)

all_visits = visits.mark_point(filled=True, size=45, color="lightgray", opacity=0.6).encode(
    x=alt.X(
        "chair_occupancy_pct:Q",
        title="Chair occupancy (%)",
        scale=alt.Scale(domain=[0, 100])
    ),
    y=alt.Y(
        "table_occupancy_pct:Q",
        title="Table occupancy (%)",
        scale=alt.Scale(domain=[0, 100])
    )
)

picked_visits = visits.transform_filter(picked_cells, empty=False).mark_point(
    filled=True,
    opacity=0.9,
    size=90
).encode(
    x="chair_occupancy_pct:Q",
    y="table_occupancy_pct:Q",
    color=alt.Color(
        "study_area:N",
        title="Study area",
        scale=alt.Scale(domain=area_order, scheme="tableau10"),
        legend=alt.Legend(titleFontSize=15, labelFontSize=13, symbolSize=150)
    ),
    shape=alt.Shape(
        "time_of_day:N",
        title="Time of day",
        scale=alt.Scale(domain=periods, range=["circle", "square", "triangle-up"]),
        legend=alt.Legend(titleFontSize=15, labelFontSize=13, symbolSize=150)
    ),
    tooltip=[
        alt.Tooltip("study_area:N", title="Study area"),
        alt.Tooltip("date:N", title="Date"),
        alt.Tooltip("day:N", title="Day"),
        alt.Tooltip("time:N", title="Time"),
        alt.Tooltip("weather:N", title="Weather"),
        alt.Tooltip("temperature:Q", title="Temperature (°F)"),
        alt.Tooltip("occupied_chairs:Q", title="Chairs taken"),
        alt.Tooltip("total_chairs:Q", title="Total chairs"),
        alt.Tooltip("occupied_tables:Q", title="Tables taken"),
        alt.Tooltip("total_tables:Q", title="Total tables"),
        alt.Tooltip("people_per_table:Q", title="People per table", format=".1f")
    ]
)

equal_line = alt.Chart(alt.Data(values=[{"x": 0, "y": 0}, {"x": 100, "y": 100}])).mark_line(
    strokeDash=[4, 4],
    color="gray"
).encode(x="x:Q", y="y:Q")

scatter_chart = (equal_line + all_visits + picked_visits).properties(
    width=330,
    height=330,
    title=alt.Title(
        "Individual visits",
        subtitle=["Gray: every visit. Colored: visits in the picked cells.",
                  "Hover over a colored point for details."]
    )
)

linked_chart = (heatmap_chart | scatter_chart).add_params(
    weather_choice
).properties(
    title="Where and when can you find an open seat?"
)

import os

os.makedirs("../visualizations/Interactive", exist_ok=True)
linked_chart.save("../visualizations/Interactive/linked_views.html")

linked_chart

alt.HConcatChart(...)

In [23]:
# Task 6: refined linked views (v2)

# Change 1. Heatmap measure toggle (chairs taken, tables taken, visits with every table taken)
# Change 2. Scatterplot shows every visit in its own color and shape instead of gray

# Change 1 data. Reshape so the heatmap can switch between three measures
# All three go the same direction, so darker always means harder to find a seat
data["all_tables_taken_pct"] = (1 - data["has_open_table"]) * 100

metric_names = {
    "chair_occupancy_pct": "Chairs taken",
    "table_occupancy_pct": "Tables taken",
    "all_tables_taken_pct": "Visits with every table taken",
}

heat_long = data.melt(
    id_vars=["visit_id", "study_area", "time_of_day", "weather"],
    value_vars=list(metric_names),
    var_name="metric",
    value_name="value",
)
heat_long["metric"] = heat_long["metric"].map(metric_names)

# Controls
picked_cells_v2 = alt.selection_point(
    fields=["study_area", "time_of_day"], on="click", clear="dblclick"
)

weather_choice_v2 = alt.param(
    value="All",
    bind=alt.binding_radio(options=["All", "Sunny", "Rain", "Cloudy"],
                           name="Weather (both views): ")
)
weather_match_v2 = (weather_choice_v2 == "All") | (alt.datum.weather == weather_choice_v2)

metric_choice = alt.param(
    value="Chairs taken",
    bind=alt.binding_radio(options=list(metric_names.values()), name="Heatmap shows: ")
)

# Heatmap
heat_v2 = alt.Chart(heat_long).transform_filter(
    weather_match_v2
).transform_filter(
    alt.datum.metric == metric_choice
).transform_aggregate(
    average="mean(value)",
    observations="valid(value)",   # skips values blanked out by fixes.csv
    groupby=["study_area", "time_of_day"]
).transform_calculate(
    percent_label=alt.expr.format(alt.datum.average, ".0f") + "%",
    visits_label=alt.expr.if_(alt.datum.observations == 1, "1 visit",
                              alt.datum.observations + " visits")
).encode(
    x=alt.X("time_of_day:O", title="Time of day", scale=alt.Scale(domain=periods),
            axis=alt.Axis(labelAngle=0, orient="top", labelFontSize=13, titleFontSize=14)),
    y=alt.Y("study_area:N", title=None, scale=alt.Scale(domain=area_order),
            axis=alt.Axis(labelFontSize=13)),
    opacity=alt.condition(picked_cells_v2, alt.value(1), alt.value(0.3))
)

cells_v2 = heat_v2.mark_rect().encode(
    color=alt.Color("average:Q", title="Average (%)",
                    scale=alt.Scale(domain=[0, 100], scheme="blues"),
                    legend=alt.Legend(orient="bottom", direction="horizontal")),
    stroke=alt.condition(picked_cells_v2, alt.value("black"), alt.value("white"), empty=False),
    strokeWidth=alt.condition(picked_cells_v2, alt.value(3), alt.value(1.5), empty=False),
    tooltip=[
        alt.Tooltip("study_area:N", title="Study area"),
        alt.Tooltip("time_of_day:O", title="Time of day"),
        alt.Tooltip("average:Q", title="Average (%)", format=".1f"),
        alt.Tooltip("observations:Q", title="Visits"),
    ]
).add_params(picked_cells_v2)

text_color = alt.condition(alt.datum.average > 60, alt.value("white"), alt.value("black"))

labels_v2 = heat_v2.mark_text(baseline="bottom", dy=-1, fontSize=16, fontWeight="bold").encode(
    text="percent_label:N", color=text_color
)
count_labels_v2 = heat_v2.mark_text(baseline="top", dy=2, fontSize=11).encode(
    text="visits_label:N", color=text_color
)

# "no visits" cells, same as before but using the new weather param
empty_base_v2 = alt.Chart(data).transform_calculate(
    matches_weather=alt.expr.if_(weather_match_v2, 1, 0)
).transform_aggregate(
    matching_visits="sum(matches_weather)", groupby=["study_area", "time_of_day"]
).transform_filter(
    alt.datum.matching_visits == 0
).encode(
    x=alt.X("time_of_day:O", scale=alt.Scale(domain=periods)),
    y=alt.Y("study_area:N", scale=alt.Scale(domain=area_order))
)
empty_cells_v2 = empty_base_v2.mark_rect(color="#f2f2f2", stroke="white", strokeWidth=1.5)
empty_labels_v2 = empty_base_v2.mark_text(fontSize=11, color="gray", text="no visits")

heatmap_v2 = (empty_cells_v2 + empty_labels_v2 + cells_v2 + labels_v2 + count_labels_v2).properties(
    width=310, height=330,
    title=alt.Title("Heatmap", subtitle="Darker means harder to find a seat")
)

# Change 2. Scatterplot, every visit in its own color and shape
visits_v2 = alt.Chart(data).transform_filter(weather_match_v2)

point_encoding = dict(
    x=alt.X("chair_occupancy_pct:Q", title="Chair occupancy (%)", scale=alt.Scale(domain=[0, 100])),
    y=alt.Y("table_occupancy_pct:Q", title="Table occupancy (%)", scale=alt.Scale(domain=[0, 100])),
    color=alt.Color("study_area:N", title="Study area",
                    scale=alt.Scale(domain=area_order, scheme="tableau10"),
                    legend=alt.Legend(titleFontSize=15, labelFontSize=13, symbolSize=150,symbolOpacity=1)),
    shape=alt.Shape("time_of_day:N", title="Time of day",
                    scale=alt.Scale(domain=periods, range=["circle", "square", "triangle-up"]),
                    legend=alt.Legend(titleFontSize=15, labelFontSize=13, symbolSize=150,symbolOpacity=1, symbolFillColor="gray")),
)

# Bottom layer. Every visit, faded but still colored, no tooltip
faded_visits = visits_v2.mark_point(filled=True, size=45, opacity=0.15).encode(**point_encoding)

# Top layer. All visits when nothing is picked, only the picked cells' visits otherwise
active_visits = visits_v2.transform_filter(picked_cells_v2).mark_point(
    filled=True, size=80, opacity=0.85
).encode(
    **point_encoding,
    tooltip=[
        alt.Tooltip("study_area:N", title="Study area"),
        alt.Tooltip("date:N", title="Date"),
        alt.Tooltip("day:N", title="Day"),
        alt.Tooltip("time:N", title="Time"),
        alt.Tooltip("weather:N", title="Weather"),
        alt.Tooltip("temperature:Q", title="Temperature (°F)"),
        alt.Tooltip("occupied_chairs:Q", title="Chairs taken"),
        alt.Tooltip("total_chairs:Q", title="Total chairs"),
        alt.Tooltip("occupied_tables:Q", title="Tables taken"),
        alt.Tooltip("total_tables:Q", title="Total tables"),
        alt.Tooltip("people_per_table:Q", title="People per table", format=".1f"),
    ]
)

equal_line_v2 = alt.Chart(alt.Data(values=[{"x": 0, "y": 0}, {"x": 100, "y": 100}])).mark_line(
    strokeDash=[4, 4], color="gray"
).encode(x="x:Q", y="y:Q")

scatter_v2 = (equal_line_v2 + faded_visits + active_visits).properties(
    width=330, height=330,
    title=alt.Title(
        "Individual visits",
        subtitle=["Click heatmap cells to highlight their visits (shift-click for more).",
                  "Hover over a point for details."]
    )
)


# Put visualization together
linked_chart_v2 = (heatmap_v2 | scatter_v2).add_params(
    weather_choice_v2, metric_choice
).properties(
    title="Where and when can you find an open seat?"
)

linked_chart_v2.save("../visualizations/Interactive/linked_views_v2.html")
linked_chart_v2

alt.HConcatChart(...)